# Function Calling 完全指南

**前置知识**: Python 基础语法、JSON 格式、函数与参数概念

**学习目标**: 掌握如何让 LLM 调用外部函数，实现 AI 与真实世界的交互

---

## 核心问题：LLM 的局限性

LLM 本质上只能「生成文本」，无法：
- 获取实时信息（天气、股价、新闻）
- 执行计算（精确数学运算）
- 操作外部系统（发邮件、查数据库）

**Function Calling 的解决方案**：让 LLM 输出「我想调用哪个函数、传什么参数」，由程序执行后返回结果。

```
用户: "北京今天多少度？"
     ↓
LLM: {"name": "get_weather", "arguments": {"city": "北京"}}  ← 结构化输出
     ↓
程序: 调用天气API → "25°C，晴"
     ↓
LLM: "北京今天25°C，天气晴朗。"
```

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
import json

# 将父目录加入路径，以便导入 src 模块
sys.path.insert(0, '..')

from src.function_calling import (
    FunctionDefinition,    # 定义函数的「说明书」
    FunctionParameter,     # 定义参数的类型和约束
    FunctionCall,          # 表示一次函数调用
    FunctionCallParser,    # 从文本中解析函数调用
    ParameterType,         # 参数类型枚举
    create_function_schema,# 从Python函数自动生成定义
)

print("导入成功！")

---

## 第一步：理解参数类型

Function Calling 使用 JSON Schema 描述参数，支持以下类型：

| 类型 | Python 对应 | 示例 |
|------|-------------|------|
| `string` | `str` | `"hello"` |
| `integer` | `int` | `42` |
| `number` | `float` | `3.14` |
| `boolean` | `bool` | `true/false` |
| `array` | `list` | `[1, 2, 3]` |
| `object` | `dict` | `{"key": "value"}` |

In [ ]:
# ============================================================
# 定义参数：FunctionParameter
# ============================================================

# 示例1：必需的字符串参数
city_param = FunctionParameter(
    name="city",                    # 参数名
    type=ParameterType.STRING,      # 类型：字符串
    description="城市名称，如'北京'", # 描述（帮助LLM理解用途）
    required=True,                  # 必需参数
)

# 示例2：可选参数 + 默认值
limit_param = FunctionParameter(
    name="limit",
    type=ParameterType.INTEGER,
    description="返回结果数量上限",
    required=False,                 # 可选参数
    default=10,                     # 默认值
)

# 示例3：枚举参数（限定取值范围）
unit_param = FunctionParameter(
    name="unit",
    type=ParameterType.STRING,
    description="温度单位",
    required=False,
    enum=["celsius", "fahrenheit"], # 只能是这两个值之一
    default="celsius",
)

# 查看生成的 JSON Schema
print("城市参数 Schema:")
print(json.dumps(city_param.to_schema(), indent=2, ensure_ascii=False))

print("\n温度单位参数 Schema:")
print(json.dumps(unit_param.to_schema(), indent=2, ensure_ascii=False))

---

## 第二步：定义完整函数

`FunctionDefinition` 组合多个参数，形成完整的函数「说明书」。

**关键点**：`description` 要写清楚函数的用途，这是 LLM 决定是否调用的依据。

In [ ]:
# ============================================================
# 定义函数：FunctionDefinition
# ============================================================

# 定义「获取天气」函数
get_weather = FunctionDefinition(
    name="get_weather",                          # 函数名（LLM调用时使用）
    description="获取指定城市的实时天气信息",      # 功能描述（越清晰越好）
    parameters=[
        FunctionParameter(
            name="city",
            type=ParameterType.STRING,
            description="城市名称，如'北京'、'上海'",
        ),
        FunctionParameter(
            name="unit",
            type=ParameterType.STRING,
            description="温度单位",
            required=False,
            enum=["celsius", "fahrenheit"],
            default="celsius",
        ),
    ],
)

# 定义「搜索」函数
search = FunctionDefinition(
    name="search",
    description="在互联网上搜索信息",
    parameters=[
        FunctionParameter(
            name="query",
            type=ParameterType.STRING,
            description="搜索关键词",
        ),
        FunctionParameter(
            name="max_results",
            type=ParameterType.INTEGER,
            description="最大返回结果数",
            required=False,
            default=5,
        ),
    ],
)

print(f"已定义函数: {get_weather.name}, {search.name}")
print(f"get_weather 参数: {[p.name for p in get_weather.parameters]}")

---

## 第三步：转换为 API 格式

不同 LLM 厂商的格式略有差异：

| 厂商 | 格式特点 |
|------|----------|
| OpenAI | 外层包裹 `{"type": "function", "function": {...}}` |
| Anthropic | 使用 `input_schema` 而非 `parameters` |

In [ ]:
# ============================================================
# 转换为 OpenAI 格式
# ============================================================
openai_format = get_weather.to_openai_schema()
print("OpenAI 格式:")
print(json.dumps(openai_format, indent=2, ensure_ascii=False))

In [ ]:
# ============================================================
# 转换为 Anthropic 格式
# ============================================================
anthropic_format = get_weather.to_anthropic_schema()
print("Anthropic 格式:")
print(json.dumps(anthropic_format, indent=2, ensure_ascii=False))

---

## 第四步：从 Python 函数自动生成

手动定义参数很繁琐，`create_function_schema` 可以从 Python 函数的类型注解自动生成。

In [ ]:
# ============================================================
# 自动生成函数定义
# ============================================================

def calculate(expression: str, precision: int = 2) -> float:
    """计算数学表达式"""
    return round(eval(expression), precision)

# 自动从函数签名生成定义
calc_def = create_function_schema(calculate)

print(f"函数名: {calc_def.name}")
print(f"描述: {calc_def.description}")
print("参数:")
for p in calc_def.parameters:
    req = "必需" if p.required else f"可选，默认={p.default}"
    print(f"  {p.name}: {p.type.value} ({req})")

---

## 第五步：解析 LLM 输出

LLM 返回的函数调用可能嵌在文本中，`FunctionCallParser` 负责提取。

**支持的格式**：
1. JSON 代码块：` ```json {...} ``` `
2. OpenAI 格式：`{"name": "...", "arguments": {...}}`
3. Anthropic 格式：`{"type": "tool_use", "name": "...", "input": {...}}`

In [ ]:
# ============================================================
# 解析函数调用
# ============================================================

# 创建解析器，传入可用的函数定义
parser = FunctionCallParser([get_weather, search])

# 模拟 LLM 输出（包含 JSON 代码块）
llm_output = '''
好的，我来帮你查询天气。

```json
{"name": "get_weather", "arguments": {"city": "北京"}}
```
'''

# 解析
calls = parser.parse(llm_output)

print(f"解析到 {len(calls)} 个函数调用:")
for call in calls:
    print(f"  函数: {call.name}")
    print(f"  参数: {call.arguments}")

In [ ]:
# ============================================================
# 解析 Anthropic 格式
# ============================================================
anthropic_output = '{"type": "tool_use", "name": "search", "input": {"query": "Python教程"}}'

calls = parser.parse(anthropic_output)
print(f"函数: {calls[0].name}")
print(f"参数: {calls[0].arguments}")

---

## 第六步：验证函数调用

LLM 可能生成错误的调用（缺参数、类型错误），需要验证。

In [ ]:
# ============================================================
# 验证函数调用
# ============================================================

# 正确的调用
valid_call = FunctionCall(name="get_weather", arguments={"city": "上海"})
errors = parser.validate(valid_call)
print(f"正确调用: {errors if errors else '✓ 验证通过'}")

# 缺少必需参数
missing_param = FunctionCall(name="get_weather", arguments={})
errors = parser.validate(missing_param)
print(f"缺少参数: {errors}")

# 未知函数
unknown_func = FunctionCall(name="unknown", arguments={})
errors = parser.validate(unknown_func)
print(f"未知函数: {errors}")

# 类型错误
wrong_type = FunctionCall(name="search", arguments={"query": "test", "max_results": "five"})
errors = parser.validate(wrong_type)
print(f"类型错误: {errors}")

---

## 实战：完整的工具调用流程

将所有组件串联，模拟真实的 Agent 工作流程。

In [ ]:
# ============================================================
# 完整示例：模拟 Agent 工具调用
# ============================================================

# 1. 定义实际执行的函数
def get_weather_impl(city: str, unit: str = "celsius") -> str:
    """模拟天气API"""
    data = {
        "北京": (25, "晴"),
        "上海": (28, "多云"),
        "广州": (32, "雨"),
    }
    temp, cond = data.get(city, (20, "未知"))
    if unit == "fahrenheit":
        temp = temp * 9/5 + 32
    return f"{city}: {cond}, {temp}{'°F' if unit == 'fahrenheit' else '°C'}"

def search_impl(query: str, max_results: int = 5) -> str:
    """模拟搜索API"""
    return f"搜索'{query}'的前{max_results}条结果..."

# 2. 函数名到实现的映射
TOOLS = {
    "get_weather": get_weather_impl,
    "search": search_impl,
}

# 3. 处理 LLM 输出的完整流程
def process_llm_output(output: str) -> str:
    """解析、验证、执行函数调用"""
    calls = parser.parse(output)
    
    if not calls:
        return "未检测到函数调用"
    
    results = []
    for call in calls:
        # 验证
        errors = parser.validate(call)
        if errors:
            results.append(f"❌ {call.name}: {errors}")
            continue
        
        # 执行
        func = TOOLS.get(call.name)
        if func:
            result = func(**call.arguments)
            results.append(f"✓ {call.name}: {result}")
        else:
            results.append(f"❌ 未找到函数: {call.name}")
    
    return "\n".join(results)

# 4. 测试
test_cases = [
    '```json\n{"name": "get_weather", "arguments": {"city": "北京"}}\n```',
    '{"name": "search", "arguments": {"query": "Python入门", "max_results": 3}}',
    '{"name": "get_weather", "arguments": {}}',  # 缺少参数
]

for i, test in enumerate(test_cases, 1):
    print(f"测试 {i}:")
    print(process_llm_output(test))
    print()

---

## 练习

定义一个 `send_email` 函数，包含 `to`（收件人）、`subject`（主题）、`body`（正文）参数。

In [ ]:
# ============================================================
# 练习：定义 send_email 函数
# ============================================================

# TODO: 补全以下代码
send_email = FunctionDefinition(
    name="send_email",
    description="发送电子邮件",
    parameters=[
        # 提示：定义 to, subject, body 三个参数
        # to: 必需，字符串
        # subject: 必需，字符串
        # body: 必需，字符串
    ],
)

# 验证：打印 OpenAI 格式
# print(json.dumps(send_email.to_openai_schema(), indent=2, ensure_ascii=False))

---

## 本节要点

| 概念 | 作用 | 关键方法 |
|------|------|----------|
| `FunctionParameter` | 定义单个参数 | `to_schema()` |
| `FunctionDefinition` | 定义完整函数 | `to_openai_schema()`, `to_anthropic_schema()` |
| `FunctionCallParser` | 解析LLM输出 | `parse()`, `validate()` |
| `create_function_schema` | 从Python函数自动生成 | - |

**下一步**: 学习 Tool Registry，管理多个工具。